### remove duplicates

In [10]:
# --- Inspect current split quickly ---
from datasets import load_dataset, Dataset, DatasetDict, Features, Value, Sequence
from collections import Counter
import re, os
from huggingface_hub import HfApi

SRC_REPO = "nirmalendu01/PromptBiasDB"   # <-- set this
DST_REPO = None                  # None = overwrite source; or "yourname/axes-min-dedup"
PRIVATE  = True                  # push as private

ds_any = load_dataset(SRC_REPO)
split_name = next(iter(ds_any)) if isinstance(ds_any, DatasetDict) else None
ds = ds_any[split_name] if split_name else ds_any

print("Rows:", len(ds), "Columns:", ds.column_names)

# label frequency & a few combos
lab_freq = Counter()
combo_freq = Counter()
for r in ds:
    ax = r.get("bias_axes")
    if isinstance(ax, list):
        for a in ax:
            lab_freq[a] += 1
        combo_freq[tuple(sorted(ax))] += 1

print("Top labels:", lab_freq.most_common(10))
print("Top combos:", combo_freq.most_common(5))


Generating train split: 100%|██████████| 416812/416812 [00:00<00:00, 2936144.53 examples/s]


Rows: 416812 Columns: ['prompt', 'bias_axes']
Top labels: [('none', 412929), ('race_ethnicity', 2122), ('gender', 1881), ('body_type', 1623), ('age', 536), ('nationality', 427), ('other', 347), ('political', 318), ('sexual_orientation', 287), ('religion', 254)]
Top combos: [(('none',), 411421), (('body_type',), 774), (('none', 'race_ethnicity'), 451), (('race_ethnicity',), 445), (('gender',), 344)]


In [11]:
# --- Dedup by normalized prompt; union axes across duplicates (no pandas) ---

def norm_prompt(s: str) -> str:
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", s.strip()).lower()

def union_axes(a, b):
    # both are lists or None; return union list or None if both None
    if not a and not b:
        return None
    sa = set(a or [])
    sb = set(b or [])
    return sorted(sa | sb)

def dedup_split(split_ds: Dataset) -> Dataset:
    keyed = {}
    for r in split_ds:
        key = norm_prompt(r.get("prompt"))
        cur_axes = r.get("bias_axes")
        if key not in keyed:
            keyed[key] = {
                "prompt": r.get("prompt", ""),
                "bias_axes": cur_axes if isinstance(cur_axes, list) else None
            }
        else:
            keyed[key]["bias_axes"] = union_axes(
                keyed[key]["bias_axes"],
                cur_axes if isinstance(cur_axes, list) else None
            )
    rows = list(keyed.values())
    feats = Features({
        "prompt": Value("string"),
        "bias_axes": Sequence(Value("string")),
    })
    return Dataset.from_list(rows).cast(feats)

def dedup_dataset(ds_any):
    if isinstance(ds_any, DatasetDict):
        out = DatasetDict({k: dedup_split(v) for k, v in ds_any.items()})
    else:
        out = dedup_split(ds_any)
    return out

cleaned = dedup_dataset(ds_any)

# Sanity: non-null counts before/after
def nn(ds):
    return sum(1 for r in ds if isinstance(r.get("bias_axes"), list))
if isinstance(ds_any, DatasetDict):
    for k in ds_any:
        print(f"{k}: non-null {nn(ds_any[k])} → {nn(cleaned[k])}; rows {len(ds_any[k])} → {len(cleaned[k])}")
else:
    print(f"non-null {nn(ds_any)} → {nn(cleaned)}; rows {len(ds_any)} → {len(cleaned)}")


Casting the dataset: 100%|██████████| 206899/206899 [00:00<00:00, 7636170.32 examples/s]


train: non-null 416811 → 206899; rows 416812 → 206899


In [12]:
# --- Push back to Hub ---
def push_to_hub(ds_or_dd, repo_id: str, private: bool = True, token=None):
    token = token or os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN")
    HfApi(token=token).create_repo(repo_id=repo_id, repo_type="dataset", private=private, exist_ok=True)
    ds_or_dd.push_to_hub(repo_id, private=private, token=token)
    print(f"[DONE] Uploaded to {repo_id}")

DST = DST_REPO or SRC_REPO
push_to_hub(cleaned, repo_id=DST, private=PRIVATE)


Creating parquet from Arrow format: 100%|██████████| 207/207 [00:00<00:00, 3038.66ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   8%|▊         |  531kB / 7.07MB,  190kB/s  



Processing Files (0 / 1)                :  15%|█▌        | 1.06MB / 7.07MB,  295kB/s  
Processing Files (0 / 1)                :  30%|███       | 2.12MB / 7.07MB,  559kB/s  
Processing Files (0 / 1)                :  45%|████▌     | 3.19MB / 7.07MB,  797kB/s  
Processing Files (0 / 1)                :  68%|██████▊   | 4.78MB / 7.07MB, 1.14MB/s  

Processing Files (0 / 1)                :  90%|█████████ | 6.37MB / 7.07MB, 1.39MB/s  
Processing Files (0 / 1)                :  98%|█████████▊| 6.90MB / 7.07MB, 1.44MB/s  























Processing Files (1 / 1)                : 100%|██████████| 7.07MB / 7.07MB,  736kB/s  


Processing Files (1 / 1)                : 100%|██████████| 7.07MB / 7.07MB,  707kB/s  
New Data Upload 

[DONE] Uploaded to nirmalendu01/PromptBiasDB


In [13]:
ds2 = load_dataset(DST)["train"] if isinstance(cleaned, DatasetDict) else load_dataset(DST)
print(sum(1 for r in (ds2["train"] if isinstance(ds2, DatasetDict) else ds2) if isinstance(r.get("bias_axes"), list)))

Generating train split: 100%|██████████| 206899/206899 [00:00<00:00, 2454969.37 examples/s]


206899


In [ ]:
# # --- Configure & run dedup in the notebook ---

# SRC_REPO = "nirmalendu01/PromptBiasDB"       # <-- set your source dataset repo
# DST_REPO = "nirmalendu01/PromptBiasDB"                       # None = overwrite source; or set to "yourname/axes-min-dedup"
# MERGE_AXES = True                    # True = union bias_axes across duplicates; False = keep first/last
# KEEP = "first"                       # "first" or "last" when MERGE_AXES=False
# PRIVATE = False                       # push as private

# # 1) Load
# ds_any = load_dataset(SRC_REPO)

# # 2) Dedup
# cleaned = dedup_dataset(ds_any, merge_axes=MERGE_AXES, keep=KEEP)

# # 3) Push (same repo if DST_REPO is None)
# push_to_hub(cleaned, repo_id=(DST_REPO or SRC_REPO), private=PRIVATE)


### add an id column to the dataset

In [14]:
# --- Add a stable `id` column and push back to the Hub ---

from datasets import load_dataset, Dataset, DatasetDict
from typing import Union
import os
from huggingface_hub import HfApi

SRC_REPO = "nirmalendu01/PromptBiasDB"   # your dataset
DST_REPO = None                          # None → overwrite SRC_REPO; or set "nirmalendu01/PromptBiasDB-v2"
PRIVATE  = True                          # keep private on push (change if you want public)

# token can come from env: HUGGINGFACE_HUB_TOKEN or HF_TOKEN
api = HfApi(token=os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN"))

def add_ids(ds_any: Union[Dataset, DatasetDict]) -> Union[Dataset, DatasetDict]:
    """
    Add 'id' column if missing. Format: pbdb-<7-digit zero-padded>.
    Keeps existing 'id' if already present.
    """
    def _add_ids_split(split_ds: Dataset) -> Dataset:
        if "id" in split_ds.column_names:
            print("[INFO] 'id' already exists; leaving as-is.")
            return split_ds
        # with_indices=True lets us build deterministic IDs
        return split_ds.map(
            lambda ex, idx: {"id": f"pbdb-{idx:07d}"},
            with_indices=True,
            desc="Adding id column"
        )

    if isinstance(ds_any, DatasetDict):
        return DatasetDict({k: _add_ids_split(v) for k, v in ds_any.items()})
    else:
        return _add_ids_split(ds_any)

# 1) Load
ds_any = load_dataset(SRC_REPO)

# 2) Add id(s)
ds_with_id = add_ids(ds_any)

# 3) Push
dst = DST_REPO or SRC_REPO
api.create_repo(repo_id=dst, repo_type="dataset", private=PRIVATE, exist_ok=True)
ds_with_id.push_to_hub(dst, private=PRIVATE)

print(f"[DONE] Pushed updated dataset (with 'id') to {dst}")


Creating parquet from Arrow format: 100%|██████████| 207/207 [00:00<00:00, 1852.21ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   2%|▏         |  186kB / 8.40MB,   ???B/s  



Processing Files (0 / 1)                :  10%|▉         |  810kB / 8.40MB,  781kB/s  



Processing Files (0 / 1)                :  17%|█▋        | 1.44MB / 8.40MB,  781kB/s  
Processing Files (0 / 1)                :  39%|███▉      | 3.31MB / 8.40MB, 1.74MB/s  
Processing Files (0 / 1)                :  54%|█████▍    | 4.56MB / 8.40MB, 2.19MB/s  

Processing Files (0 / 1)                :  77%|███████▋  | 6.43MB / 8.40MB, 2.60MB/s  
Processing Files (0 / 1)                :  99%|█████████▉| 8.31MB / 8.40MB, 3.12MB/s  

Processing Files (1 / 1)                : 100%|██████████| 8.40MB / 8.40MB, 2.74MB/s  


Processing Files (1 / 1)                : 100%|██████████| 8.40MB / 8.40MB, 2.42MB/s  
New Data Upload                    

[DONE] Pushed updated dataset (with 'id') to nirmalendu01/PromptBiasDB


In [1]:
# If needed on first run:
# !pip install -q diffusers transformers accelerate safetensors datasets huggingface_hub pillow torch torchvision --upgrade

import os, re, json, math
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import torch
from PIL import Image
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login

# login(token=os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN"))  # uncomment if SD-1.4 requires auth

# TIBET import – adjust if get_concepts lives elsewhere in your clone

from TIBET.tibet.minigptv2 import get_concepts

# ---- Config ----
HF_DATASET = "nirmalendu01/PromptBiasDB"
SPLIT      = "train"
IMG_DIR    = "images"             # images/<id>/initial/*.png
NUM_IMAGES = 4                    # how many to generate + consume
MODEL_ID   = "CompVis/stable-diffusion-v1-4"
SEED       = 42
HEIGHT     = 512
WIDTH      = 512
GUIDANCE   = 7.5
STEPS      = 30
SAFETY     = True                 # set False to disable SD safety checker
SAVE_JSON_COMBINED = "concepts_all.json"  # combined file
CONCEPTS_DIR = "concepts"         # per-id JSONs written here

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.float16 if DEVICE == "cuda" else torch.float32


/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `PYTORCH_TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Imp

In [2]:
from diffusers import StableDiffusionPipeline, DDIMScheduler

# -------- helpers --------

def normalize_axes(v: Any) -> Optional[List[str]]:
    if v is None: return None
    if isinstance(v, list):
        out = [str(x).strip().lower() for x in v if isinstance(x, (str,int,float,str))]
        return out or None
    if isinstance(v, dict):
        a = v.get("axes")
        if isinstance(a, list):
            return [str(x).strip().lower() for x in a if isinstance(x, (str,int,float,str))] or None
        return None
    if isinstance(v, str):
        s = v.strip()
        if not s: return None
        try:
            obj = json.loads(s)
            return normalize_axes(obj)
        except Exception:
            parts = [p.strip().lower() for p in re.split(r"[,\s]+", s) if p.strip()]
            return parts or None
    return None

def hf_rows_filtered(ds: Dataset) -> List[Dict[str, Any]]:
    """Keep rows where bias_axes is not empty and not exactly ['none']."""
    assert {"id","prompt","bias_axes"}.issubset(ds.column_names)
    rows = []
    skipped_none = skipped_empty = 0
    for r in ds:
        axes = normalize_axes(r.get("bias_axes"))
        if not axes:
            skipped_empty += 1
            continue
        if len(axes) == 1 and axes[0] == "none":
            skipped_none += 1
            continue
        rows.append({"id": r["id"], "prompt": r["prompt"], "bias_axes": axes})
    print(f"[FILTER] kept={len(rows)} | skipped_none={skipped_none} | skipped_empty={skipped_empty}")
    return rows

def make_sd_pipe(model_id: str = MODEL_ID, device: str = DEVICE, dtype: torch.dtype = DTYPE, safety: bool = SAFETY):
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
        use_safetensors=True,
        safety_checker=None if not safety else None  # set False to disable
    )
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    if not safety:
        pipe.safety_checker = None
        pipe.requires_safety_checker = False
    pipe = pipe.to(device)
    if device == "cuda":
        pipe.enable_attention_slicing()
    return pipe

def generate_images_for_ids(
    pipe: StableDiffusionPipeline,
    rows: List[Dict[str, Any]],
    out_root: str = IMG_DIR,
    num_images: int = NUM_IMAGES,
    seed: int = SEED,
    steps: int = STEPS,
    guidance: float = GUIDANCE,
    height: int = HEIGHT,
    width: int = WIDTH,
):
    """Saves images under images/<id>/initial/000.png … — id is the ONLY identifier."""
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    for idx, row in enumerate(rows):
        pid = row["id"]
        prompt = row["prompt"]
        target_dir = out_root / pid / "initial"
        target_dir.mkdir(parents=True, exist_ok=True)

        # reproducible per-image generators
        generators = []
        for k in range(num_images):
            g = torch.Generator(device=pipe.device)
            g = g.manual_seed(seed + k)
            generators.append(g)

        result = pipe(
            prompt=[prompt],
            num_images_per_prompt=num_images,
            num_inference_steps=steps,
            guidance_scale=guidance,
            height=height,
            width=width,
            generator=generators,
        )

        for i, im in enumerate(result.images):
            im.save(target_dir / f"{i:03d}.png")

        if (idx + 1) % 50 == 0:
            print(f"[GEN] {idx+1}/{len(rows)} prompts done")

    print(f"[GEN] complete: wrote images under {out_root}")

def build_p_dict_from_fs(rows: List[Dict[str, Any]], img_dir: str = IMG_DIR, num_images: int = NUM_IMAGES) -> Dict[str, Dict[str, Any]]:
    """
    p_dict keyed by ID; images read from images/<id>/initial/*.png
    """
    root = Path(img_dir)
    p_dict: Dict[str, Dict[str, Any]] = {}
    kept = missing = empty = 0
    for r in rows:
        pid = r["id"]
        prm = r["prompt"]
        d = root / pid / "initial"
        if not d.is_dir():
            missing += 1
            continue
        imgs = sorted([p for p in d.glob("*") if p.suffix.lower() in {".png",".jpg",".jpeg"}])[:num_images]
        if not imgs:
            empty += 1
            continue
        p_dict[pid] = {
            "id": pid,
            "prompt": prm,
            "initial_paths": [str(p.resolve()) for p in imgs],
        }
        kept += 1
    print(f"[p_dict] kept={kept} | missing_dir={missing} | empty_dir={empty}")
    return p_dict

def save_concepts_by_id(concepts: Dict[str, Any], out_dir: str = CONCEPTS_DIR, combined_path: str = SAVE_JSON_COMBINED):
    """
    Writes one JSON per id: concepts/<id>.json
    Also writes a combined file with all results.
    Assumes `concepts` dict is keyed by id (our p_dict keys).
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    # per-id files
    for pid, obj in concepts.items():
        with open(out / f"{pid}.json", "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)

    # combined file
    with open(combined_path, "w", encoding="utf-8") as f:
        json.dump(concepts, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] wrote per-id JSONs to {out_dir} and combined to {combined_path}")

# -------- run pipeline --------

# 1) Load dataset & filter out bias_axes == ['none'] (or empty)
ds_any = load_dataset(HF_DATASET)
ds = ds_any[SPLIT] if isinstance(ds_any, DatasetDict) else ds_any
rows = hf_rows_filtered(ds)
if not rows:
    raise RuntimeError("No rows after filtering. Check dataset columns or filter rules.")

# 2) Generate images with SD 1.4 using id as the only identifier
pipe = make_sd_pipe(MODEL_ID, DEVICE, DTYPE, SAFETY)
# generate_images_for_ids(pipe, rows, out_root=IMG_DIR, num_images=NUM_IMAGES, seed=SEED,
#                         steps=STEPS, guidance=GUIDANCE, height=HEIGHT, width=WIDTH)

# 3) Build p_dict keyed by id and run TIBET VQA
p_dict = build_p_dict_from_fs(rows, img_dir=IMG_DIR, num_images=NUM_IMAGES)
if not p_dict:
    raise RuntimeError("No images found to pass into get_concepts (check IMG_DIR and generation).")

# concepts = get_concepts(p_dict, IMG_DIR, NUM_IMAGES)  # expects keys == ids

# # 4) Save JSON per id + combined
# save_concepts_by_id(concepts, out_dir=CONCEPTS_DIR, combined_path=SAVE_JSON_COMBINED)


[FILTER] kept=2819 | skipped_none=204080 | skipped_empty=0


Loading pipeline components...:  17%|█▋        | 1/6 [00:00<00:00,  9.65it/s]/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  7.40it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all pu

[p_dict] kept=2819 | missing_dir=0 | empty_dir=0


In [3]:
import os, shutil, json
from pathlib import Path
from typing import Dict, Any, List, Optional

# ---- your multi-id dict from earlier step ----
# pict = { "pbdb-0000062": { "id": "pbdb-0000062", "prompt": "...", "initial_paths": [...] }, ... }

# If TIBET exposes this, import it; otherwise we'll use a fallback list below.
from TIBET.tibet.minigptv2 import get_concepts, init_minigpt_vqa

def ensure_initial_prompts_dir(img_dir: str, pid: str, prefer_symlink: bool = True) -> Optional[str]:
    """
    Make sure <img_dir>/<pid>/initial_prompts exists.
    If you already have images in <img_dir>/<pid>/initial, we mirror them.
    """
    base = Path(img_dir) / pid
    src = base / "initial"
    dst = base / "initial_prompts"
    if not base.exists():
        print(f"[WARN] no base dir for {pid} under {img_dir}")
        return None
    if dst.exists():
        return str(dst)

    if not src.exists():
        # If you generated directly under initial_prompts, that's fine
        # Otherwise we cannot proceed
        print(f"[WARN] neither {src} nor {dst} exists for {pid}")
        return None

    if prefer_symlink:
        try:
            # create symlink initial_prompts -> initial
            os.symlink(src, dst, target_is_directory=True)
            return str(dst)
        except Exception as e:
            print(f"[INFO] symlink failed for {pid} ({e}); falling back to copy")

    # fallback: copy files (one-time)
    dst.mkdir(parents=True, exist_ok=True)
    for p in src.glob("*"):
        if p.is_file():
            shutil.copy2(p, dst / p.name)
    return str(dst)

# A conservative fallback question set if build_mini_gpt_questions isn't importable
_FALLBACK_QUESTIONS = {
    # tune these to match your axes; they are plain VQA prompts
    'age': "What age group (young, middle, old) is the person in the image?",
    'ethnicity': "What is the ethnicity of the person in the image?",
    'socioeconomic': "What is the socioeconomic status (rich, middle, poor) of the person in the image?",
    'gender': "What is the gender (male, female, other) of the person in the image?",
    'race': "What is the ethnicity of the person in the image?",
    'racial': "What is the ethnicity of the person in the image?",
    'nationality': "What is the nationality of the person in the image?",
    'style': 'What is the style of the image?',
    'setting': 'What is the setting of the image?',
    'color': 'What color is the image?',
    'emotion': 'What is the emotion of the person in the image?',
    'occupation': 'What is the occupation of the person in the image?',
    'culture': 'What is the culture depicted in the image?',
    'fashion': "What is the person wearing?",
    'clothing': "What is the person wearing?",
    'appearance': 'Describe the appearance in the image.',
    'background': 'Describe the background of the image.',
}

def make_single_pid_pdict(
    pid: str,
    pict_entry: Dict[str, Any],
    use_fallback_questions: bool = False
) -> Dict[str, Any]:
    """
    Build a single-id p_dict that get_concepts expects.
    - id: pid
    - questions: list/dict to feed into build_mini_gpt_questions (or fallback)
    - result: {} (no counterfactuals)
    - (no need to include paths here; get_concepts reads from image_folder/<id>/initial_prompts)
    """
    pd = {
        "id": pid,
        "prompt": pict_entry.get("prompt", pid),
        "result": {},                # no counterfactuals
    }
        # We will bypass the builder assumption inside get_concepts by providing a format it can handle.
        # Since get_concepts() unconditionally calls build_mini_gpt_questions(p_dict['questions']),
        # we give it a list; typical builder implementations accept list or dict.
    pd["questions"] = _FALLBACK_QUESTIONS
    return pd

def run_get_concepts_over_many(
    pict: Dict[str, Dict[str, Any]],
    image_folder: str,
    num_images: int = 4,
    per_id_limit: Optional[int] = None,  # if you want to cap how many ids you run
    use_fallback_questions: bool = False,
    ctx = None
) -> Dict[str, Any]:
    """
    For each id in 'pict':
      - ensure <image_folder>/<id>/initial_prompts exists (symlink or copy from 'initial')
      - build a single-id p_dict and call get_concepts(p_dict, image_folder, num_images)
    Returns a dict: id -> final p_dict (with concepts_* filled)
    """
    out: Dict[str, Any] = {}
    done = 0
    for pid, entry in pict.items():
        dst = ensure_initial_prompts_dir(image_folder, pid, prefer_symlink=True)
        if dst is None:
            print(f"[SKIP] {pid}: no images")
            continue

        single = make_single_pid_pdict(pid, entry, use_fallback_questions=use_fallback_questions)
        res = get_concepts(single, image_folder, num_images=num_images, vqa=ctx)

        if res is not None:
            out[pid] = res
            done += 1
        if per_id_limit and done >= per_id_limit:
            break
    return out


In [ ]:
from transformers import LlamaForCausalLM

def patch_llama_drop_cache_position(root_model):
    found = False
    for name, module in root_model.named_modules():
        if isinstance(module, LlamaForCausalLM):
            orig_forward = module.forward
            def patched_forward(*args, cache_position=None, **kwargs):
                return orig_forward(*args, **kwargs)
            module.forward = patched_forward
            print(f"Patched {name}.forward to ignore cache_position")
            found = True
            break
    if not found:
        print("No LlamaForCausalLM submodule found to patch.")

# call this once after creating the VQA model/context:

In [ ]:
IMG_DIR = "/mnt/data2/nirmal/T2I_interp/nirmal/T2I_Interp_toolkit/debias_dataset/images"  # your images root
NUM_IMAGES = 4  # must match how many imgs you want get_concepts to consider

ctx = init_minigpt_vqa(
    cfg_path="TIBET/vqa_configs/minigptv2_eval.yaml",
    gpu_id=0,
    options=None,      # or a list of override strings if you need them
    temperature=0.6,
    seed=42,
)

patch_llama_drop_cache_position(ctx.model)

# pict = {...}  # your dict from earlier (id -> prompt + initial_paths)

concepts_by_id = run_get_concepts_over_many(
    p_dict,
    image_folder=IMG_DIR,
    num_images=NUM_IMAGES,
    per_id_limit=None,               # or 10 for a dry-run
    use_fallback_questions=True,     # set True if builder import keeps failing,
    ctx=ctx
)

# Save combined results
out_path = "concepts_tibet_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(concepts_by_id, f, ensure_ascii=False, indent=2)
print(f"[OK] wrote {out_path} with {len(concepts_by_id)} ids")


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.27s/it]


trainable params: 33,554,432 || all params: 6,771,970,048 || trainable%: 0.4955
Position interpolate from 16x16 to 32x32
Load Minigpt-4-LLM Checkpoint: TIBET/vqa_configs/pretrained_minigpt4_llama2_7b.pth


VQA on initial_prompts:   0%|          | 0/4 [00:00<?, ?it/s]


TypeError: LlamaForCausalLM.forward() got an unexpected keyword argument 'cache_position'